# Studio 2 — Klasifikasi Sampah: Kasus → Final

Notebook ini menjalankan **seluruh alur** dalam satu kali *Run all*:

| Bagian | Isi | Hasil |
|---|---|---|
| **A · Kasus** | Latih dengan validasi `my_val` (15%), pilih epoch terbaik berdasarkan **Macro-F1** | skor lokal, kurva belajar, F1 per kelas, galeri kesalahan |
| **B · Final** | Latih ulang konfigurasi yang sama di **seluruh** data (+~440 gambar) | model final |
| **C · Paket** | Bentuk folder `submission/` yang sudah lolos `evaluate.py` resmi | `submission_A_tervalidasi.zip`, `submission_B_final.zip` |

**Cara pakai**
1. **Runtime → Change runtime type → T4 GPU**
2. Ubah sel **KONFIGURASI** (cukup `DATA_SRC`)
3. **Runtime → Run all** — total ±30–45 menit di T4

Notebook ini adalah **bengkel**, bukan yang dinilai. Yang dinilai asisten adalah folder `submission/` di repo GitHub. Simpan notebook ini di `notebooks/` repo sebagai kode pelatihan (bagian *Cara Reproduksi* di README).

In [ ]:
!pip -q install -U "timm>=1.0"
import os, sys, json, shutil, subprocess, time
import torch, timm
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print(f"torch {torch.__version__} | timm {timm.__version__} | GPU: {gpu or 'TIDAK ADA'}")
if gpu is None:
    print("PERINGATAN: tanpa GPU training sangat lambat. Runtime -> Change runtime type -> T4 GPU, lalu Run all lagi.")


def run(cmd):
    """Jalankan perintah shell, tampilkan output langsung, dan HENTIKAN notebook bila gagal."""
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        if "Warning" in line or "warnings.warn" in line:
            continue
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"Perintah gagal (exit {p.returncode}). Baca pesan error di atas.")

## KONFIGURASI — satu-satunya sel yang perlu diubah

**`DATA_SRC`** adalah folder `train/` yang berisi 5 subfolder kelas. Cara menemukannya:
1. Di Google Drive, buka folder dataset dari asisten → klik kanan → **Organize → Add shortcut → My Drive**.
2. Setelah sel *Dataset* me-mount Drive, buka panel 📁 di kiri Colab → `drive/MyDrive/...` → klik kanan folder `train` → **Copy path**.

Parameter lain sudah berisi nilai awal yang masuk akal. Saat bereksperimen, **ubah satu parameter saja per run** dan ganti `NAME` agar hasil lama tidak tertimpa.

In [ ]:
# ===================== UBAH DI SINI =====================
DATA_SRC = '/content/drive/MyDrive/STUDIO2/train'   # folder train/ (5 subfolder kelas)
BACKBONE = 'convnext_tiny.fb_in22k_ft_in1k'         # nama model timm
EPOCHS   = 15            # epoch Bagian A
LR       = 1e-4          # LR backbone (head = 10x)
BATCH    = 32
AUG      = 'strong'      # 'strong' atau 'light'
SEED     = 42
NAME     = 'cnxt_t'      # nama eksperimen -> runs/A_<NAME>, runs/B_<NAME>_full
FINAL_EPOCHS = None      # None = otomatis = epoch terbaik Bagian A
SAVE_TO_DRIVE = '/content/drive/MyDrive/studio2_runs'
EXTRA_ARGS = ''          # lanjutan, mis. '--img-size 288'
# ========================================================
WORK = '/content/work' if IN_COLAB else os.path.abspath('work')
RUN_A = f'runs/A_{NAME}'
RUN_B = f'runs/B_{NAME}_full'
COMMON = f'--backbone {BACKBONE} --lr {LR} --batch-size {BATCH} --aug {AUG} --seed {SEED} {EXTRA_ARGS}'
print('kerja di', WORK)

## 1. Dataset

Dataset disalin dari Drive ke disk lokal Colab, karena membaca ribuan file kecil langsung dari Drive sangat lambat. Dataset **tidak pernah** masuk repo.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

os.makedirs(os.path.join(WORK, 'submission'), exist_ok=True)
os.chdir(WORK)
DATA = os.path.join(WORK, 'data', 'train')
if not os.path.isdir(DATA):
    if not os.path.isdir(DATA_SRC):
        raise FileNotFoundError(f'DATA_SRC tidak ditemukan: {DATA_SRC}\n-> cek path di panel file kiri, lalu ubah sel KONFIGURASI.')
    t0 = time.time()
    shutil.copytree(DATA_SRC, DATA)
    print(f'disalin dalam {time.time() - t0:.0f} s')

EXPECTED = {'botol_plastik': 541, 'gelas_plastik': 572, 'kertas_kardus': 395, 'saset_kemasan': 758, 'styrofoam': 698}
for c, n_exp in EXPECTED.items():
    d = os.path.join(DATA, c)
    if not os.path.isdir(d):
        raise FileNotFoundError(f'folder kelas hilang: {d} — apakah DATA_SRC menunjuk ke folder train/?')
    n = len([f for f in os.listdir(d) if f.lower().endswith('.jpg')])
    print(f'{c:15s} {n:4d}' + ('' if n == n_exp else f'   (dokumentasi: {n_exp})'))
SOURCES_CSV = os.path.join(os.path.dirname(DATA_SRC.rstrip('/')), 'train_sources.csv')
print('train_sources.csv:', 'ada' if os.path.exists(SOURCES_CSV) else 'tidak ditemukan (opsional)')

## 2. Kode

Tiga sel berikut menulis file yang **sama persis** dengan yang ada di repo: `evaluate.py` (salinan skrip resmi), `submission/model.py`, dan `train.py`. Kalau kamu mengubah `train.py` di repo, salin perubahannya ke sini juga, supaya kode yang di-commit sama dengan kode yang menghasilkan model.

In [ ]:
%%writefile evaluate.py
"""Skrip evaluasi resmi Studio 2 — Klasifikasi Sampah (IF25-40401).

Skrip INI JUGA yang dipakai asisten dosen untuk menghitung skor pada validation set
tersembunyi. Pastikan submission kelompok Anda lolos skrip ini sebelum tenggat.

Pemakaian (uji pada split validasi buatan Anda sendiri dari data train):
    python evaluate.py --submission ./submission --data ./my_val

    --submission  folder berisi model.py dan model.pth
    --data        folder berstruktur <data>/<nama_kelas>/*.jpg
    --out         (opsional) simpan hasil ke file JSON

Kontrak submission (folder submission/):
    model.py   wajib mendefinisikan:
                 get_model()     -> torch.nn.Module (arsitektur saja, TANPA mengunduh bobot)
                 get_transform() -> callable: PIL.Image (RGB) -> torch.Tensor [3, H, W]
    model.pth  state_dict hasil torch.save(model.state_dict(), "model.pth")
               Output model: logits berukuran [batch, 5] dengan urutan kelas = CLASSES di bawah.
"""
import argparse
import importlib.util
import json
import sys
import time
from pathlib import Path

import torch
from PIL import Image

CLASSES = ["botol_plastik", "gelas_plastik", "kertas_kardus", "saset_kemasan", "styrofoam"]
EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}


def load_submission(sub_dir, device):
    sub_dir = Path(sub_dir).resolve()
    spec = importlib.util.spec_from_file_location("submission_model", sub_dir / "model.py")
    mod = importlib.util.module_from_spec(spec)
    sys.path.insert(0, str(sub_dir))  # izinkan model.py meng-import file pendukung di folder yang sama
    spec.loader.exec_module(mod)
    for fn in ("get_model", "get_transform"):
        if not hasattr(mod, fn):
            raise SystemExit(f"[GAGAL] model.py tidak mendefinisikan {fn}()")
    model = mod.get_model()
    state = torch.load(sub_dir / "model.pth", map_location="cpu", weights_only=True)
    model.load_state_dict(state, strict=True)
    model.to(device).eval()
    return model, mod.get_transform()


def list_images(data_dir):
    items = []
    for ci, cls in enumerate(CLASSES):
        d = Path(data_dir) / cls
        if not d.is_dir():
            raise SystemExit(f"[GAGAL] folder kelas tidak ditemukan: {d}")
        items += [(p, ci) for p in sorted(d.iterdir()) if p.suffix.lower() in EXTS]
    return items


def metrics(y_true, y_pred, k=len(CLASSES)):
    cm = [[0] * k for _ in range(k)]
    for t, p in zip(y_true, y_pred):
        cm[t][p] += 1
    f1s = []
    for c in range(k):
        tp = cm[c][c]
        fp = sum(cm[r][c] for r in range(k)) - tp
        fn = sum(cm[c]) - tp
        prec = tp / (tp + fp) if tp + fp else 0.0
        rec = tp / (tp + fn) if tp + fn else 0.0
        f1s.append(2 * prec * rec / (prec + rec) if prec + rec else 0.0)
    acc = sum(cm[c][c] for c in range(k)) / max(1, len(y_true))
    return {"macro_f1": sum(f1s) / k, "accuracy": acc,
            "per_class_f1": dict(zip(CLASSES, f1s)), "confusion_matrix": cm}


@torch.no_grad()
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--submission", required=True)
    ap.add_argument("--data", required=True)
    ap.add_argument("--batch-size", type=int, default=32)
    ap.add_argument("--out")
    a = ap.parse_args()

    device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
    model, tf = load_submission(a.submission, device)
    items = list_images(a.data)
    print(f"device={device}  gambar={len(items)}")

    y_true, y_pred, t0 = [], [], time.time()
    for i in range(0, len(items), a.batch_size):
        chunk = items[i:i + a.batch_size]
        x = torch.stack([tf(Image.open(p).convert("RGB")) for p, _ in chunk]).to(device)
        logits = model(x)
        if logits.ndim != 2 or logits.shape[1] != len(CLASSES):
            raise SystemExit(f"[GAGAL] output model harus [batch, {len(CLASSES)}], didapat {tuple(logits.shape)}")
        y_pred += logits.argmax(1).cpu().tolist()
        y_true += [c for _, c in chunk]

    m = metrics(y_true, y_pred)
    m["n_images"] = len(items)
    m["seconds"] = round(time.time() - t0, 1)
    print(f"\nMacro-F1 : {m['macro_f1']:.4f}\nAkurasi  : {m['accuracy']:.4f}")
    for c, f in m["per_class_f1"].items():
        print(f"  F1 {c:<15} {f:.4f}")
    print("Confusion matrix (baris = label asli, kolom = prediksi):")
    for c, row in zip(CLASSES, m["confusion_matrix"]):
        print(f"  {c:<15} {row}")
    if a.out:
        Path(a.out).write_text(json.dumps(m, indent=2))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile submission/model.py
"""Submission Studio 2 — model generik berbasis timm.

Arsitektur & preprocessing TIDAK ditulis langsung di sini, tetapi dibaca dari
`config.json` di folder yang sama (dibuat otomatis oleh train.py). Alasannya:
konfigurasi selalu "ikut" bersama bobotnya, sehingga model.py, config.json,
dan model.pth tidak mungkin saling tidak cocok tanpa ketahuan.

Isi folder submission/ yang wajib di-commit:
    model.py      (file ini — tidak perlu diubah)
    config.json   (backbone, ukuran input, mean/std)
    model.pth     (state_dict)

Kontrak evaluate.py:
    get_model()     -> nn.Module, output logits [batch, 5], TANPA mengunduh bobot
    get_transform() -> PIL.Image (RGB) -> Tensor [3, H, W]
Urutan kelas: 0 botol_plastik, 1 gelas_plastik, 2 kertas_kardus, 3 saset_kemasan, 4 styrofoam
"""
import json
from pathlib import Path

import timm
from torchvision import transforms

NUM_CLASSES = 5
CONFIG_PATH = Path(__file__).resolve().parent / "config.json"


def load_config(path=CONFIG_PATH):
    if not Path(path).exists():
        raise FileNotFoundError(f"config.json tidak ditemukan di {Path(path).parent} — "
                                "salin config.json dari folder run bersama model.pth.")
    return json.loads(Path(path).read_text())


def build_model(backbone, pretrained=False):
    """Dipakai train.py (pretrained=True) dan get_model() (pretrained=False).
    Fungsi yang sama -> nama key state_dict pasti identik."""
    return timm.create_model(backbone, pretrained=pretrained, num_classes=NUM_CLASSES)


def build_eval_transform(img_size, crop_pct, mean, std):
    return transforms.Compose([
        transforms.Resize(int(round(img_size / crop_pct))),  # sisi pendek
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])


def get_model():
    cfg = load_config()
    return build_model(cfg["backbone"], pretrained=False)


def get_transform():
    cfg = load_config()
    return build_eval_transform(cfg["img_size"], cfg["crop_pct"], cfg["mean"], cfg["std"])

In [ ]:
%%writefile train.py
"""Training Studio 2 — fine-tuning backbone timm, checkpoint dipilih berdasarkan Macro-F1.

Jalankan dari root repo:
    python train.py --data ../data/train --backbone convnext_tiny.fb_in22k_ft_in1k --epochs 15 --name cnxt_e15
    python evaluate.py --submission runs/cnxt_e15/submission --data ./my_val

Setiap run menghasilkan runs/<name>/:
    submission/          model.py + config.json + model.pth -> folder submission lengkap
    log.csv              metrik per epoch (loss, Macro-F1, akurasi, F1 per kelas)
    val_predictions.csv  prediksi checkpoint terbaik di my_val, diurutkan dari yang paling "salah"
    args.json            semua argumen run (reproduksibilitas)

Mode --full: latih di my_train + my_val dengan jumlah epoch tetap (untuk model Final).

Perbedaan utama dari train_baseline.py:
  1. Checkpoint dipilih berdasarkan Macro-F1 (metrik penilaian), bukan akurasi.
  2. Metrik & jalur load memakai fungsi dari evaluate.py -> angka identik dengan penilai.
  3. Mean/std/ukuran input diambil dari pretrained_cfg backbone, disimpan di config.json.
  4. Warmup + cosine per-step, LR head lebih besar dari backbone, AMP di GPU.
  5. Bobot > 95 MB otomatis disimpan fp16 agar muat di GitHub (< 100 MB).
"""
import argparse
import contextlib
import csv
import json
import math
import random
import shutil
import time
from pathlib import Path

import numpy as np
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import ConcatDataset, DataLoader
from PIL import Image
from torchvision import datasets, transforms

import evaluate                       # metrik + jalur load yang SAMA dengan penilai
from submission import model as sub   # build_model / build_eval_transform

CLASSES = evaluate.CLASSES
FP16_EXPORT_MB = 95


# ----------------------------------------------------------------------------- data
def make_local_split(src, dst, frac, seed):
    """Sama persis dengan train_baseline.py -> split my_val identik bila seed/frac sama.
    Hanya dibuat sekali; kalau my_val/ sudah ada, split lama dipakai ulang."""
    dst_tr, dst_va = Path(dst) / "my_train", Path(dst) / "my_val"
    if dst_va.exists():
        return dst_tr, dst_va
    rng = random.Random(seed)
    for cls_dir in sorted(Path(src).iterdir()):
        if not cls_dir.is_dir():
            continue
        files = sorted(cls_dir.glob("*.jpg"))
        rng.shuffle(files)
        n_va = int(len(files) * frac)
        for i, f in enumerate(files):
            out = (dst_va if i < n_va else dst_tr) / cls_dir.name
            out.mkdir(parents=True, exist_ok=True)
            shutil.copy2(f, out / f.name)
    return dst_tr, dst_va


def build_train_transform(cfg, aug):
    size = cfg["img_size"]
    to_tensor = [transforms.ToTensor(), transforms.Normalize(cfg["mean"], cfg["std"])]
    if aug == "light":  # setara baseline
        return transforms.Compose([
            transforms.RandomResizedCrop(size, scale=(0.6, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(0.2, 0.2, 0.2),
            *to_tensor,
        ])
    return transforms.Compose([  # "strong"
        transforms.RandomResizedCrop(size, scale=(0.4, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.TrivialAugmentWide(),
        *to_tensor,
        transforms.RandomErasing(p=0.25),
    ])


# ----------------------------------------------------------------------------- utils
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def model_config(model, img_size):
    pc = model.pretrained_cfg
    return {
        "backbone": None,  # diisi pemanggil
        "img_size": img_size or pc["input_size"][-1],
        "crop_pct": pc.get("crop_pct") or 0.875,
        "mean": [float(v) for v in pc["mean"]],
        "std": [float(v) for v in pc["std"]],
        "timm_version": timm.__version__,
        "torch_version": torch.__version__,
    }


@torch.no_grad()
def predict(model, dl, device):
    model.eval()
    probs, ys, loss_sum = [], [], 0.0
    for x, y in dl:
        logits = model(x.to(device)).float()
        loss_sum += F.cross_entropy(logits, y.to(device), reduction="sum").item()
        probs.append(logits.softmax(1).cpu())
        ys.append(y)
    p, y = torch.cat(probs), torch.cat(ys)
    return y.tolist(), p, loss_sum / len(y)


def export_submission(state, cfg, out_dir):
    """Tulis folder submission lengkap: model.py + config.json + model.pth."""
    out_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(Path(sub.__file__), out_dir / "model.py")
    (out_dir / "config.json").write_text(json.dumps(cfg, indent=2))
    mb = sum(v.numel() * v.element_size() for v in state.values()) / 1e6
    if mb > FP16_EXPORT_MB:
        state = {k: (v.half() if v.is_floating_point() else v) for k, v in state.items()}
        print(f"bobot {mb:.0f} MB > {FP16_EXPORT_MB} MB -> disimpan fp16")
    torch.save(state, out_dir / "model.pth")
    return (out_dir / "model.pth").stat().st_size / 1e6


@torch.no_grad()
def official_check(sub_dir, data_dir, device):
    """Muat submission PERSIS seperti evaluate.py (strict=True, weights_only=True)."""
    model, tf = evaluate.load_submission(sub_dir, device)
    items = evaluate.list_images(data_dir)
    y_true, y_pred = [], []
    for i in range(0, len(items), 32):
        chunk = items[i:i + 32]
        x = torch.stack([tf(Image.open(p).convert("RGB")) for p, _ in chunk]).to(device)
        y_pred += model(x).argmax(1).cpu().tolist()
        y_true += [c for _, c in chunk]
    return evaluate.metrics(y_true, y_pred)


# ----------------------------------------------------------------------------- main
def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--data", required=True, help="folder train/ asli dari dataset")
    ap.add_argument("--split-dir", default=".", help="lokasi my_train/ dan my_val/")
    ap.add_argument("--backbone", default="convnext_tiny.fb_in22k_ft_in1k", help="nama model timm")
    ap.add_argument("--img-size", type=int, default=None, help="default: ukuran pretrained backbone")
    ap.add_argument("--epochs", type=int, default=15)
    ap.add_argument("--batch-size", type=int, default=32)
    ap.add_argument("--lr", type=float, default=1e-4, help="LR backbone")
    ap.add_argument("--head-lr-mult", type=float, default=10.0, help="LR head = lr x nilai ini")
    ap.add_argument("--wd", type=float, default=0.05)
    ap.add_argument("--warmup-epochs", type=float, default=1.0)
    ap.add_argument("--label-smoothing", type=float, default=0.1)
    ap.add_argument("--aug", choices=["light", "strong"], default="strong")
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--workers", type=int, default=2)
    ap.add_argument("--name", default=None, help="nama folder di runs/")
    ap.add_argument("--full", action="store_true", help="latih di my_train + my_val (model Final)")
    ap.add_argument("--no-pretrained", action="store_true", help="tanpa bobot ImageNet (uji/scratch)")
    a = ap.parse_args()

    seed_everything(a.seed)
    device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
    name = a.name or f"{a.backbone.split('.')[0]}_{time.strftime('%m%d-%H%M')}"
    run = Path("runs") / name
    run.mkdir(parents=True, exist_ok=True)
    (run / "args.json").write_text(json.dumps(vars(a), indent=2))

    # ---- model + konfigurasi preprocessing (sumber kebenaran tunggal)
    model = sub.build_model(a.backbone, pretrained=not a.no_pretrained)
    cfg = model_config(model, a.img_size)
    cfg["backbone"] = a.backbone
    model.to(device)
    print(f"device={device}  backbone={a.backbone}  input={cfg['img_size']}  mean={cfg['mean']}")

    # ---- data
    tr_dir, va_dir = make_local_split(a.data, a.split_dir, 0.15, 42)
    train_tf = build_train_transform(cfg, a.aug)
    eval_tf = sub.build_eval_transform(cfg["img_size"], cfg["crop_pct"], cfg["mean"], cfg["std"])
    tr_ds = datasets.ImageFolder(tr_dir, train_tf)
    va_ds = datasets.ImageFolder(va_dir, eval_tf)
    assert tr_ds.classes == CLASSES == va_ds.classes, f"urutan kelas salah: {tr_ds.classes}"
    if a.full:
        tr_ds = ConcatDataset([tr_ds, datasets.ImageFolder(va_dir, train_tf)])
    print(f"train={len(tr_ds)}  val={len(va_ds)}  mode={'FULL (tanpa validasi)' if a.full else 'normal'}")

    kw = dict(num_workers=a.workers, pin_memory=device == "cuda", persistent_workers=a.workers > 0)
    tr_dl = DataLoader(tr_ds, a.batch_size, shuffle=True, drop_last=True, **kw)
    va_dl = DataLoader(va_ds, a.batch_size, shuffle=False, **kw)

    # ---- optimizer: head belajar lebih cepat daripada backbone pretrained
    head_ids = {id(p) for p in model.get_classifier().parameters()}
    opt = torch.optim.AdamW([
        {"params": [p for p in model.parameters() if id(p) not in head_ids], "lr": a.lr},
        {"params": [p for p in model.parameters() if id(p) in head_ids], "lr": a.lr * a.head_lr_mult},
    ], weight_decay=a.wd)
    total = a.epochs * len(tr_dl)
    warm = max(1, int(a.warmup_epochs * len(tr_dl)))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm
                                              else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    loss_fn = nn.CrossEntropyLoss(label_smoothing=a.label_smoothing)
    use_amp = device == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    amp_ctx = (lambda: torch.autocast("cuda", dtype=torch.float16)) if use_amp else contextlib.nullcontext

    # ---- training loop
    header = ["epoch", "lr", "train_loss", "val_loss", "macro_f1", "acc"] + [f"f1_{c}" for c in CLASSES] + ["sec"]
    log_f = open(run / "log.csv", "w", newline="")
    log = csv.writer(log_f)
    log.writerow(header)
    best_f1, best_loss, best_ep, best_state, best_probs = -1.0, float("inf"), 0, None, None

    for ep in range(1, a.epochs + 1):
        t0 = time.time()
        model.train()
        loss_sum, n = 0.0, 0
        for x, y in tr_dl:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with amp_ctx():
                loss = loss_fn(model(x), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            loss_sum += loss.item() * len(y)
            n += len(y)

        y_true, probs, val_loss = predict(model, va_dl, device)
        m = evaluate.metrics(y_true, probs.argmax(1).tolist())
        sec = time.time() - t0
        log.writerow([ep, f"{opt.param_groups[0]['lr']:.2e}", f"{loss_sum / n:.4f}", f"{val_loss:.4f}",
                      f"{m['macro_f1']:.4f}", f"{m['accuracy']:.4f}"]
                     + [f"{m['per_class_f1'][c]:.4f}" for c in CLASSES] + [f"{sec:.0f}"])
        log_f.flush()

        # Macro-F1 lebih tinggi menang; bila seri, pilih val_loss lebih rendah.
        # Mode --full tidak punya validasi yang jujur -> selalu ambil epoch terakhir.
        improved = a.full or m["macro_f1"] > best_f1 or (m["macro_f1"] == best_f1 and val_loss < best_loss)
        if improved:
            best_f1, best_loss, best_ep, best_probs = m["macro_f1"], val_loss, ep, probs
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(f"epoch {ep:2d}/{a.epochs}  train_loss={loss_sum / n:.4f}  val_loss={val_loss:.4f}  "
              f"macroF1={m['macro_f1']:.4f}  acc={m['accuracy']:.4f}  {sec:.0f}s {'*' if improved else ''}")
    log_f.close()

    # ---- prediksi checkpoint terbaik -> bahan analisis error & label noise
    with open(run / "val_predictions.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["file", "label", "pred", "p_label", "p_pred", "benar"])
        rows = []
        for (path, lab), p in zip(va_ds.samples, best_probs):
            pr = int(p.argmax())
            rows.append([Path(path).relative_to(va_dir).as_posix(), CLASSES[lab], CLASSES[pr],
                         round(float(p[lab]), 4), round(float(p[pr]), 4), int(pr == lab)])
        w.writerows(sorted(rows, key=lambda r: r[3]))  # paling "yakin salah" di atas

    # ---- ekspor + cek lewat jalur evaluate.py
    sub_dir = run / "submission"
    size_mb = export_submission(best_state, cfg, sub_dir)
    check = official_check(sub_dir, va_dir, device)
    (run / "best.json").write_text(json.dumps({
        "epoch": best_ep, "epochs": a.epochs, "full": a.full,
        "macro_f1_train_loop": round(best_f1, 4), "macro_f1_evaluate": round(check["macro_f1"], 4),
        "accuracy_evaluate": round(check["accuracy"], 4), "per_class_f1": check["per_class_f1"],
        "confusion_matrix": check["confusion_matrix"], "model_mb": round(size_mb, 1),
    }, indent=2))
    print("\n" + "=" * 70)
    print(f"run              : {run}")
    print(f"checkpoint       : epoch {best_ep}  (Macro-F1 my_val saat training = {best_f1:.4f})")
    print(f"cek evaluate.py  : Macro-F1 = {check['macro_f1']:.4f}  akurasi = {check['accuracy']:.4f}")
    print(f"model.pth        : {size_mb:.1f} MB  |  timm {timm.__version__}")
    if a.full:
        print("CATATAN: mode --full -> my_val ikut dilatih, skor di atas hanya cek 'bisa jalan'.")
    print(f"\nPromosikan ke submission/:  copy {sub_dir}/* -> submission/")
    print("=" * 70)


if __name__ == "__main__":
    main()

---
# Bagian A — Kasus: eksperimen dengan validasi

`train.py` memotong 15% data per kelas menjadi `my_val/` (seed 42, sama dengan `train_baseline.py`), melatih di sisanya, dan menyimpan epoch dengan **Macro-F1 tertinggi**. Di akhir, folder submission hasilnya dimuat **persis seperti cara asisten** (`strict=True`, `weights_only=True`).

In [ ]:
run(f'python -u train.py --data {DATA} --split-dir . --epochs {EPOCHS} --name A_{NAME} {COMMON}')

### Skor dengan skrip evaluasi resmi
Perintah ini sama dengan yang dijalankan asisten, hanya data validasinya berbeda.

In [ ]:
run(f'python -u evaluate.py --submission {RUN_A}/submission --data my_val')

### Kurva belajar
Cara membaca:
- **Loss**: kalau `val_loss` naik sementara `train_loss` terus turun, model mulai *overfit*.
- **Macro-F1**: lihat di epoch berapa puncaknya. Puncak di tengah lalu turun tajam juga tanda overfit.
- **F1 per kelas**: kelas dengan garis paling bawah adalah target perbaikan berikutnya.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

CLASSES = ['botol_plastik', 'gelas_plastik', 'kertas_kardus', 'saset_kemasan', 'styrofoam']
SERIES = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4']   # urutan tetap
INK, MUTED, GRID = '#0b0b0b', '#52514e', '#e5e4e0'

log = pd.read_csv(f'{RUN_A}/log.csv')
best_a = json.load(open(f'{RUN_A}/best.json'))
be = best_a['epoch']

def style(ax, title, ylabel):
    ax.set_title(title, loc='left', color=INK, fontsize=11)
    ax.set_xlabel('epoch', color=MUTED); ax.set_ylabel(ylabel, color=MUTED)
    ax.grid(color=GRID, lw=0.8); ax.set_axisbelow(True)
    for s in ('top', 'right'): ax.spines[s].set_visible(False)
    for s in ('left', 'bottom'): ax.spines[s].set_color(GRID)
    ax.tick_params(colors=MUTED)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
ax = axes[0]
for col, c, lab in [('train_loss', SERIES[0], 'train'), ('val_loss', SERIES[1], 'validasi')]:
    ax.plot(log.epoch, log[col], color=c, lw=2, marker='o', ms=4, label=lab)
style(ax, 'Loss', 'cross-entropy'); ax.legend(frameon=False, labelcolor=MUTED)

ax = axes[1]
ax.plot(log.epoch, log.macro_f1, color=SERIES[0], lw=2, marker='o', ms=4)
bv = log.loc[log.epoch == be, 'macro_f1'].item()
ax.plot([be], [bv], 'o', ms=9, mfc='white', mec=SERIES[0], mew=2)
ax.annotate(f'terbaik: epoch {be}\n{bv:.4f}', (be, bv), textcoords='offset points', xytext=(-10, -32),
            ha='right' if be > len(log) / 2 else 'left', color=INK, fontsize=9)
style(ax, 'Macro-F1 (my_val)', 'Macro-F1')

ax = axes[2]
for c, col in zip(CLASSES, SERIES):
    ax.plot(log.epoch, log[f'f1_{c}'], color=col, lw=2, label=c)
style(ax, 'F1 per kelas (my_val)', 'F1')
ax.legend(frameon=False, fontsize=8, labelcolor=MUTED, loc='lower right')
plt.tight_layout(); plt.show()

print(f"Checkpoint terpilih: epoch {be}/{best_a['epochs']}  |  Macro-F1 (evaluate.py) = {best_a['macro_f1_evaluate']}")
print('F1 per kelas:', {k: round(v, 3) for k, v in best_a['per_class_f1'].items()})
log.round(4)

### Galeri kesalahan
Prediksi salah di `my_val`, diurutkan dari yang **paling yakin salah** (probabilitas label asli paling kecil). Untuk setiap gambar, tanyakan: **apakah model yang salah, atau label-nya yang salah?** Kalau ada `train_sources.csv`, judul halaman sumber ikut ditampilkan sebagai petunjuk.

In [ ]:
from math import ceil
from PIL import Image

pred = pd.read_csv(f'{RUN_A}/val_predictions.csv')
if os.path.exists(SOURCES_CSV):
    titles = pd.read_csv(SOURCES_CSV).set_index('file')['title']
    pred['title'] = pred.file.map(titles)
err = pred[pred.benar == 0]
print(f'{len(err)} salah dari {len(pred)} gambar my_val')

show = err.head(12)
if len(show):
    cols = 4
    rows = ceil(len(show) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4.4 * rows), squeeze=False)
    for ax in axes.flat:
        ax.axis('off')
    for ax, (_, r) in zip(axes.flat, show.iterrows()):
        ax.imshow(Image.open(os.path.join('my_val', r.file)).convert('RGB'))
        ax.set_title(f'label: {r.label}\nprediksi: {r.pred} ({r.p_pred:.2f})', fontsize=9, color=INK)
    plt.tight_layout(); plt.show()
    pd.set_option('display.max_colwidth', 80)
    display(show)

### Refleksi sebelum lanjut ke Final
Jawab dulu tiga pertanyaan ini dan catat di README:
1. Kelas mana yang F1-nya terendah? Pasangan kelas mana yang paling sering tertukar di confusion matrix?
2. Dari 12 kesalahan teratas, berapa yang menurutmu **label-nya** salah, bukan model-nya?
3. Apakah ada tanda overfit? Di epoch berapa?

Kalau ingin mencoba konfigurasi lain (backbone, `AUG`, `LR`, `EPOCHS`), ubah sel KONFIGURASI dan jalankan ulang Bagian A dengan `NAME` baru **sebelum** lanjut ke Bagian B.

---
# Bagian B — Final: latih di seluruh data

Di Bagian A, 15% data (~440 gambar) hanya dipakai untuk validasi. Untuk model final, data itu ikut dilatih supaya model melihat lebih banyak variasi.

**Konsekuensinya:** tidak ada lagi validasi yang jujur untuk memilih epoch. Jadi jumlah epoch ditetapkan **dari Bagian A** (`FINAL_EPOCHS` = epoch terbaik). Jadwal LR cosine akan menyesuaikan diri dengan jumlah epoch itu.

In [ ]:
if FINAL_EPOCHS is None:
    FINAL_EPOCHS = best_a['epoch']
if FINAL_EPOCHS < best_a['epochs'] / 2:
    print(f'PERHATIAN: epoch terbaik ({FINAL_EPOCHS}) jauh sebelum akhir ({best_a["epochs"]}) -> kemungkinan overfit. '
          'Pertimbangkan AUG/regularisasi lebih kuat di Bagian A.')
print('FINAL_EPOCHS =', FINAL_EPOCHS)
run(f'python -u train.py --data {DATA} --split-dir . --epochs {FINAL_EPOCHS} --full --name B_{NAME}_full {COMMON}')

---
# Bagian C — Paket submission

Notebook membuat **dua** paket. Keduanya berisi folder `submission/` lengkap (`model.py`, `config.json`, `model.pth`, `requirements.txt` dengan versi timm yang di-pin), dan masing-masing **sudah diuji dengan `evaluate.py` resmi**.

| Paket | Isi | Kapan dipakai |
|---|---|---|
| `submission_A_tervalidasi` | checkpoint terbaik Bagian A | **Preview 1** (skornya bisa dicocokkan dengan validasi lokal) |
| `submission_B_final` | model data penuh Bagian B | **Preview 2**, lalu **Final** bila skornya ≥ Preview 1 |

Strateginya: skor preview dari asisten adalah satu-satunya pembanding yang **tidak bias**. Submit A di Preview 1 dan B di Preview 2, lalu bandingkan.

In [ ]:
PAKET = 'paket'
shutil.rmtree(PAKET, ignore_errors=True)
zips = []
for tag, run_dir in [('A_tervalidasi', RUN_A), ('B_final', RUN_B)]:
    root = os.path.join(PAKET, f'submission_{tag}')
    dst = os.path.join(root, 'submission')
    os.makedirs(dst)
    for f in ('model.py', 'config.json', 'model.pth'):
        shutil.copy2(os.path.join(run_dir, 'submission', f), dst)
    with open(os.path.join(dst, 'requirements.txt'), 'w') as fh:
        fh.write(f'timm=={timm.__version__}\n')
    print(f'\n===== cek paket {tag} dengan evaluate.py resmi =====')
    run(f'python -u evaluate.py --submission {dst} --data my_val')
    if tag.startswith('B'):
        print('(B ikut dilatih dengan my_val -> skor di atas hanya membuktikan paket bisa dimuat.)')
    zips.append(shutil.make_archive(root, 'zip', root))

print()
for z in zips:
    print(f'{z}  ({os.path.getsize(z) / 1e6:.1f} MB)')

if IN_COLAB:
    dst = os.path.join(SAVE_TO_DRIVE, f'{NAME}_{time.strftime("%m%d-%H%M")}')
    os.makedirs(dst, exist_ok=True)
    for z in zips:
        shutil.copy2(z, dst)
    for r in (RUN_A, RUN_B):
        shutil.copytree(r, os.path.join(dst, os.path.basename(r)), ignore=shutil.ignore_patterns('*.pth', '__pycache__'))
    print('tersimpan di Drive:', dst)
    from google.colab import files
    for z in zips:
        files.download(z)

---
## Setelah notebook selesai (di laptop)

1. Ekstrak `submission_A_tervalidasi.zip`, lalu **timpa** folder `submission/` di repo dengan isinya (4 file).
2. Uji, persis seperti yang dilakukan asisten:
   ```
   python evaluate.py --submission ./submission --data ./my_val
   ```
   Skornya harus sama dengan Bagian A, karena `my_val` di laptop identik (seed 42).
3. Pastikan file yang akan masuk aman: `git add --dry-run .` (tidak boleh ada `.jpg` atau `runs/`).
4. Commit dan push, misalnya: `git commit -m "Preview 1: ConvNeXt-T A, macroF1 lokal 0.xxxx"`.
5. Catat skor lokal dan nama run di tabel **Hasil** README.

Saat Preview 2, ulangi langkah yang sama dengan `submission_B_final.zip`.